In [ ]:
"""
THIS CODE POSTPROCESSES NPZ PARTICLE DATA AND SAVES AS HITS

TANVIR M. SAURAV

BUSHFIRE RESEARCH GROUP
UNSW CANBERRA

"""

import os
os.chdir('F:/npzs/')
import numpy as np
import particlepost as par

case_id = 15 # 5-10-15

cases, rows, cols, xspace, yspace = par.init_case_params(case_id)

clx, cly, clz = 15.0, 15.0, 8.0  # cube dims
start_x = 400.0
start_y = (150.0 - rows * cly - (rows - 1) * yspace) / 2.0

polygs = par.create_polygon_array(start_x, start_y, rows, cols, clx, cly, xspace, yspace)

# Define output directory for hit data
output_dir = "./polygon_hit_data"
os.makedirs(output_dir, exist_ok=True)

for case in cases:
    print(f"Starting {case}...")

    # Load data for this case
    data = np.load(case + ".npz", mmap_mode='r')[case][:, :, :3]

    # Run hit counting
    losing_positions, cumulative_counts = par.count_polygon_hits(data, polygs, clz)

    # Convert to structured array
    losing_positions_array = np.array(
        losing_positions,
        dtype=[('timestep', 'i4'), ('particle_index', 'i4'), ('polygon_index', 'i4')]
    )

    # Save per case
    np.savez_compressed(
        os.path.join(output_dir, f"hits_{case}.npz"),
        losing_positions=losing_positions_array,
        cumulative_counts=cumulative_counts
    )

    print(f"Saved hit data for {case} in {output_dir}/hits_{case}.npz")

print("All cases processed and saved.")

